<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day08-discussion-1.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 8, Part 1 discussion — Regression or classification?

The book page treated helix fraction as a **regression** target: predict
a number between 0 and 1. But you could also ask a yes/no question:
*is this protein mostly helical* (helix fraction > 0.5)? That is a
**classification** problem built from the same data.

**Question:** is it better to (a) train a classifier on the yes/no label
directly, or (b) train a regression model on the full number and
threshold its prediction at 0.5? Make a prediction before you run the
cells.

In [1]:
import numpy as np
import requests
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import StratifiedKFold
from scipy.stats import pearsonr

PDB_IDS = ['21XG', '38LJ', '9WRU', '9NVT', '9XZ9', '7ILS', '9RYL', '9TLR', '32TC', '13DZ', '9TCX', '9YBX', '9N2Y', '29OL', '9PNX', '9YMO', '9HWE', '9RPQ', '9RR6', '9W2A', '21LO', '9RK7', '9OU3', '9P39', '9T4I', '9VOM', '9W28', '9RFG', '9V5C', '9RQP', '9I4O', '9QWL', '9QYO', '9WM8', '9QRF', '9X6R', '9TZG', '9QGV', '9U97', '9YK5', '9IC9', '9XPD', '9DJF', '9DJR', '9NK4', '9NCE', '9I1B', '9RQQ', '7IPN', '9LCS', '9Z70', '9KZN', '9SI4', '9P45', '9FD0']
HELIX_FORMERS = "EMALKFQ"   # Chou-Fasman helix propensity >= 1.1

def fetch_helix_fraction(pdbid):
    lo = pdbid.lower()
    r = requests.get(f"https://www.ebi.ac.uk/pdbe/api/pdb/entry/molecules/{lo}", timeout=15)
    r.raise_for_status()
    mol = [m for m in r.json()[lo] if m.get("molecule_type") == "polypeptide(L)" and m.get("sequence")][0]
    seq, chain_id, length = mol["sequence"], mol["in_chains"][0], mol["length"]
    r = requests.get(f"https://www.ebi.ac.uk/pdbe/api/pdb/entry/secondary_structure/{lo}", timeout=15)
    r.raise_for_status()
    helices = []
    for molecule in r.json()[lo]["molecules"]:
        for chain in molecule["chains"]:
            if chain["chain_id"] == chain_id:
                helices = chain["secondary_structure"].get("helices", [])
    n_helix = sum(h["end"]["residue_number"] - h["start"]["residue_number"] + 1 for h in helices)
    return seq, n_helix / length

helix_seqs, helix_frac = [], []
for pdbid in PDB_IDS:
    s, h = fetch_helix_fraction(pdbid)
    helix_seqs.append(s); helix_frac.append(h)
helix_frac = np.array(helix_frac)
former_frac = np.array([sum(s.count(a) for a in HELIX_FORMERS) / len(s) for s in helix_seqs])
print(f"{len(helix_frac)} proteins; helix fraction ranges {helix_frac.min():.2f}-{helix_frac.max():.2f}")
r, p = pearsonr(former_frac, helix_frac)
print(f"Pearson r between helix-former content and helix fraction: {r:.2f} (p = {p:.1e})")

55 proteins; helix fraction ranges 0.03-0.84
Pearson r between helix-former content and helix fraction: 0.71 (p = 1.3e-09)


In [2]:
X = former_frac.reshape(-1, 1)
is_helical = (helix_frac > 0.5).astype(int)
print(f"{is_helical.sum()} of {len(is_helical)} proteins are mostly helical (helix fraction > 0.5)")

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
acc_clf, acc_reg = [], []
for tr, te in folds.split(X, is_helical):
    clf = LogisticRegression().fit(X[tr], is_helical[tr])              # (a) classify directly
    acc_clf.append((clf.predict(X[te]) == is_helical[te]).mean())
    reg = LinearRegression().fit(X[tr], helix_frac[tr])                 # (b) regress, then threshold
    acc_reg.append(((reg.predict(X[te]) > 0.5).astype(int) == is_helical[te]).mean())
print(f"(a) logistic regression on the yes/no label: 5-fold CV accuracy {np.mean(acc_clf):.3f}")
print(f"(b) linear regression, thresholded at 0.5:   5-fold CV accuracy {np.mean(acc_reg):.3f}")

16 of 55 proteins are mostly helical (helix fraction > 0.5)
(a) logistic regression on the yes/no label: 5-fold CV accuracy 0.709
(b) linear regression, thresholded at 0.5:   5-fold CV accuracy 0.818


## What does the yes/no label throw away?

Two proteins with helix fractions 0.49 and 0.05 get the same label,
while 0.49 and 0.51 get different ones. How many proteins sit close to
the threshold, where the label is almost arbitrary?

In [3]:
near = np.abs(helix_frac - 0.5) < 0.1
print(f"proteins within 0.1 of the threshold: {near.sum()} of {len(helix_frac)}")
reg_all = LinearRegression().fit(X, helix_frac)
resid = helix_frac - reg_all.predict(X)
print(f"regression residual std (whole dataset): {resid.std():.3f}")
print("measured helix fractions near the threshold:", np.round(np.sort(helix_frac[near]), 2))

proteins within 0.1 of the threshold: 11 of 55
regression residual std (whole dataset): 0.154
measured helix fractions near the threshold: [0.41 0.44 0.44 0.44 0.49 0.5  0.5  0.52 0.54 0.57 0.57]


**Discuss:**

1. Which approach won here, and by how much? With 55 proteins and 5
   folds, is the difference larger than the noise you would expect
   (Day 8, Part 2)?
2. The regression model's typical error (residual std) is about the same
   size as the ±0.1 window around the threshold. What does that mean for
   the proteins in that window?
3. Name a biological quantity where you would *prefer* to keep the
   number (regression), and one where the category is the natural
   label.